In [68]:
import re
import nltk
from nltk.corpus import stopwords, wordnet

In [67]:
import nltk
#nltk.data.path.append('/Users/margauxlanglois/nltk_data')  # remplace <ton_nom> par ton nom d'utilisateur

# Tu peux maintenant utiliser :
from nltk.corpus import stopwords
stop_words_fr = set(stopwords.words('french'))

In [69]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
 
example_sent = """This is a sample sentence,
                  showing off the stop words filtration."""
 
word_tokens = word_tokenize(example_sent, language='french')
# converts the words in word_tokens to lower case and then checks whether 
#they are present in stop_words or not
filtered_sentence = [w for w in word_tokens if not w.lower() in stop_words_fr]
#with no lower case conversion
filtered_sentence = []
 
for w in word_tokens:
    if w not in stop_words_fr:
        filtered_sentence.append(w)
 
print(word_tokens)
print(filtered_sentence)

LookupError: 
**********************************************************************
  Resource [93mpunkt_tab[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('punkt_tab')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtokenizers/punkt_tab/french/[0m

  Searched in:
    - '/Users/margauxlanglois/nltk_data'
    - '/Users/margauxlanglois/Documents/SNCF/s2025p2-mobile-app-incidents/env/nltk_data'
    - '/Users/margauxlanglois/Documents/SNCF/s2025p2-mobile-app-incidents/env/share/nltk_data'
    - '/Users/margauxlanglois/Documents/SNCF/s2025p2-mobile-app-incidents/env/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
    - 'stopwords'
    - './stopwords'
    - '/Users/margauxlanglois/Documents/SNCF/s2025p2-mobile-app-incidents/RegExp/stopwords'
    - '/Users/margauxlanglois/nltk_data'
    - 

In [46]:
stop_words_fr = {
    "le", "la", "les", "un", "une", "des", "du", "de", "d", "et", "en", "à", "au", "aux",
    "ce", "cet", "cette", "ces", "dans", "par", "pour", "avec", "sur", "sous", "chez",
    "qui", "que", "quoi", "dont", "où", "ne", "pas", "plus", "moins", "est", "sont",
    "était", "été", "être", "avoir", "a", "ont", "mais", "ou", "si", "donc", "or", "ni", "car",
    "comme", "lorsque", "quand", "tandis", "alors", "ainsi", "afin", "se", "sa", "son",
    "leur", "leurs", "nos", "notre", "votre", "vos", "mon", "ma", "mes", "toi", "moi", "on", "1", "2", "3", "4", "5", "6", "7", "8", "9", "0",
}


In [5]:
with open("../incidents_listes/incidents_categories.json", "r", encoding="utf-8") as f:
    incident_data = json.load(f)

with open("../incidents_listes/incidents_arbo_comp.json", "r", encoding="utf-8") as f:
    incident_arbo = json.load(f)

with open("../tests/messages.json", "r", encoding="utf-8") as f:
    messages = json.load(f)

In [62]:



stop_words_en = set(stopwords.words('english'))

def normalize(text):
    """Nettoie et découpe un texte en mots utiles (sans ponctuation ni stopwords)"""
    words = re.sub(r'\W+', ' ', text.lower()).split()
    return set(word for word in words if word and word not in stop_words_en)

def get_synonyms(word):
    """Retourne un ensemble de synonymes pour un mot (en anglais)"""
    synonyms = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            synonym = lemma.name().replace('_', ' ').lower()
            if synonym != word:
                synonyms.add(synonym)
    return synonyms

def build_label_synonyms(tree):
    """Construit un dictionnaire label -> mots clés + synonymes"""
    label_synonyms = {}

    def traverse(subtree):
        for label, child in subtree.items():
            label_words = normalize(label)
            all_synonyms = set(label_words)
            for word in label_words:
                all_synonyms.update(get_synonyms(word))
            label_synonyms[label] = all_synonyms
            if isinstance(child, dict):
                traverse(child)

    traverse(tree)
    return label_synonyms

def search_category_by_keywords(tree, description, label_synonyms, path=[]):
    """Recherche récursive avec synonymes"""
    words_in_description = normalize(description)

    for label, subtree in tree.items():
        label_words = label_synonyms.get(label, set())

        if words_in_description & label_words:
            print(f"Match sur {label} (avec mots {words_in_description & label_words})")
            new_path = path + [label]
            if isinstance(subtree, dict) and subtree:
                deeper = search_category_by_keywords(subtree, description, label_synonyms, new_path)
                if deeper:
                    return deeper
            return new_path
        else:
            if isinstance(subtree, dict):
                deeper = search_category_by_keywords(subtree, description, label_synonyms, path)
                if deeper:
                    return deeper
    return path if path else None


In [63]:
description = messages["1"]
print("Description de l'incident :", description)
result = search_category_by_keywords(incident_arbo, description)
print("Catégorie détectée :", result)


Description de l'incident : La lunette des toilettes glisse dans le wagon 3.


TypeError: search_category_by_keywords() missing 1 required positional argument: 'label_synonyms'